# Model Training

### Setup MLFlow

In [ ]:
# # SETUP BLOCK (Run once per project lifetime)
# import mlflow
# import os

# # 1. Define Absolute Paths
# project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
# db_uri = f"sqlite:///{os.path.join(project_root, 'mlflow.db')}"
# artifact_uri = "file:///" + os.path.join(project_root, "mlruns").replace("\\", "/")

# # 2. Connect
# mlflow.set_tracking_uri(db_uri)

# # 3. Create (Safe Check)
# if not mlflow.get_experiment_by_name("WaterQuality"):
#     mlflow.create_experiment("WaterQuality", artifact_location=artifact_uri)

In [ ]:
import mlflow
import os

# --- ENVIRONMENT TOGGLE ---
ENV = 'local' # Change to 'snowflake' when you deploy to the cloud

if ENV == 'local':
    import config_local as config
else:
    import config_snowflake as config

# Set tracking URI dynamically
mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment("WaterQuality")

### Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import shap
import time

from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold

# Regression Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, StackingRegressor

warnings.filterwarnings("ignore")

### Load Data

In [ ]:
# Dynamically load data based on environment toggle
df = config.load_data()

# 1. Assign LORO Regions (Required for CV Folds)
def assign_region(row):
    if row['Latitude'] >= -30.0:
        return 'Northern_Bulk'
    elif row['Longitude'] < 22.5:
        return 'Western_Cape'
    else:
        return 'Eastern_Cape'

df['Region'] = df.apply(assign_region, axis=1)

# 2. Drop Spatial Anchors (Prevent Spatial Leakage)
SPATIAL_COLS = ['Latitude', 'Longitude', 'Sample Date']
df = df.drop(columns=SPATIAL_COLS)

TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
META_COLS = ['Region']

# 3. Unsupervised Pre-Filter
initial_features = [col for col in df.columns if col not in TARGET_COLS + META_COLS]
print(f"Initial features from DE: {len(initial_features)}")

# Variance Check
var_filter = VarianceThreshold(threshold=0.0)
var_filter.fit(df[initial_features].fillna(0))
non_constant_features = [f for f, s in zip(initial_features, var_filter.get_support()) if s]
print(f"Dropped {len(initial_features) - len(non_constant_features)} zero-variance features.")

# Collinearity Check
CORR_THRESHOLD = 0.90 
corr_matrix = df[non_constant_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper.columns if any(upper[column] > CORR_THRESHOLD)]
print(f"Dropped {len(to_drop_corr)} highly correlated features.")

UNIVERSAL_FEATURES = [f for f in non_constant_features if f not in to_drop_corr]
print(f"Final lean feature count ready for baseline training: {len(UNIVERSAL_FEATURES)}")

### Define Model Hyperparameters & Preprocessor

In [ ]:
data_params = {
    "validation_strategy": "Leave-One-Region-Out (LORO) Spatial CV",
    "folds": "3 (Northern_Bulk, Western_Cape, Eastern_Cape)",
    "pipeline_architecture": "ColumnTransformer (Fit strictly INSIDE CV loops)",
    "imputation_strategy": "Median via Train Folds Only",
    "target_handling": "Raw Values (Disabled log1p to match benchmark)" 
}

def get_preprocessor():
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median', add_indicator=False)),
        ('scaler', StandardScaler()),
    ])
    return ColumnTransformer(
        transformers=[('num', numeric_transformer, UNIVERSAL_FEATURES)],
        remainder='drop'
    )

In [ ]:
# 1. Define the Base Models (The "Council")
base_estimators = [
    ('etr_shallow', ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ('etr_deep', ExtraTreesRegressor(n_estimators=100, max_depth=25, min_samples_split=5, random_state=42, n_jobs=-1))
]

# 2. Define the Meta-Learner (The "Judge")
meta_learner = KernelRidge(alpha=0.00001, kernel='poly', degree=2, coef0=0.25)

# 3. Build the Stack
stacked_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    passthrough=False 
)

# 4. Final Council
models = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
    "StackedModel": stacked_model
}

In [ ]:
# # 1. Define the Base Models (The "Council")
# # We use ExtraTrees with slight variations to ensure they learn differently
# base_estimators = [
#     ('etr_shallow', ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
#     ('etr_deep', ExtraTreesRegressor(n_estimators=100, max_depth=25, min_samples_split=5, random_state=42, n_jobs=-1))
# ]

# # 2. Define the Meta-Learner (The "Smoother")
# # The winner used Ridge with a polynomial kernel. We start with standard Ridge as a safe baseline.
# meta_learner = KernelRidge(alpha=0.00001, kernel='poly', degree=2, coef0=0.25)

# # 3. Build the Stack
# stacked_model = StackingRegressor(
#     estimators=base_estimators,
#     final_estimator=meta_learner,
#     passthrough=False # Set to True if you want Ridge to also see the raw features
# )

# # 4. Add to your pipeline
# models = {
#     "Ridge": Ridge(alpha=1.0),
#     "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
#     "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1),
#     "StackedModel": stacked_model # <--- Ready for the new data!
# }

### Model Training & Evaluation

p.s. We may need to split `numeric_transformer` into two: one that imputes medians for weather/satellite data, and one that imputes 0 for GIS buffers.

In [ ]:
results = []
regions = df['Region'].unique()

# Calculate total iterations for the outer progress bar
total_runs = len(TARGET_COLS) * len(models)

# 1. Outer Progress Bar (Tracks the 9 combinations)
with tqdm(total=total_runs, desc="Overall Progress") as pbar:
    for target in TARGET_COLS:
        for model_name, base_model in models.items():
            run_name = f"{model_name}_{target.replace(' ', '')}_LORO"
            
            with mlflow.start_run(run_name=run_name):
                # Use tqdm.write instead of print to prevent progress bar corruption
                tqdm.write(f"\n--- Cross-Validating {run_name} ---")
                
                # Arrays to store Out-Of-Fold (OOF) predictions for honest scoring
                oof_y_true = []
                oof_y_pred = []
                
                cv_start_time = time.time()
                
                # ==========================================
                # 1. The LORO Cross-Validation Loop
                # ==========================================
                # 2. Inner Progress Bar (Tracks the 3 spatial folds, disappears when done)
                for holdout_region in tqdm(regions, desc=f"Folds for {model_name}", leave=False):
                    tqdm.write(f"  Holding out {holdout_region}...")
                    
                    # Split Data geographically
                    train_df = df[df['Region'] != holdout_region]
                    test_df = df[df['Region'] == holdout_region]
                    
                    X_train_raw = train_df[UNIVERSAL_FEATURES]
                    y_train = train_df[target]
                    X_test_raw = test_df[UNIVERSAL_FEATURES]
                    y_test = test_df[target]
                    
                    # Dynamically fit preprocessor strictly on the training folds
                    fold_preprocessor = get_preprocessor()
                    X_train = fold_preprocessor.fit_transform(X_train_raw)
                    X_test = fold_preprocessor.transform(X_test_raw)
                    
                    # Train & Predict
                    model = clone(base_model)
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                    
                    # Store predictions for the overall OOF score
                    oof_y_true.extend(y_test)
                    oof_y_pred.extend(y_pred)
                    
                cv_time = time.time() - cv_start_time
                
                # ==========================================
                # 2. Honest OOF Scoring & Logging
                # ==========================================
                oof_y_true = np.array(oof_y_true)
                oof_y_pred = np.array(oof_y_pred)
                
                metrics = {
                    "rmse": np.sqrt(mean_squared_error(oof_y_true, oof_y_pred)),
                    "mae": mean_absolute_error(oof_y_true, oof_y_pred),
                    "r2": r2_score(oof_y_true, oof_y_pred),
                    "cv_time_sec": cv_time
                }
                
                mlflow.log_params(data_params)
                mlflow.log_param("target", target)
                mlflow.log_param("model", model_name)
                mlflow.log_metrics(metrics)
                
                # Actual vs Predicted Plot (Using OOF Predictions)
                fig, ax = plt.subplots(figsize=(6, 6))
                ax.scatter(oof_y_true, oof_y_pred, alpha=0.4, color='teal')
                min_val = min(oof_y_true.min(), oof_y_pred.min())
                max_val = max(oof_y_true.max(), oof_y_pred.max())
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
                ax.set_xlabel('Actual Values (All Folds)')
                ax.set_ylabel('Predicted Values (OOF)')
                ax.set_title(f'True Generalization - {run_name}')
                mlflow.log_figure(fig, 'actual_vs_predicted_oof.png')
                plt.close(fig)

                # ==========================================
                # 3. Retrain on 100% of Data for Submission
                # ==========================================
                tqdm.write("  Retraining final model on 100% of data...")
                final_preprocessor = get_preprocessor()
                X_all = final_preprocessor.fit_transform(df[UNIVERSAL_FEATURES])
                y_all = df[target]
                
                final_model = clone(base_model)
                final_model.fit(X_all, y_all)
                
                # Save the final Preprocessor artifact (overwrites with the 100% trained version)
                config.save_artifacts(final_preprocessor, final_model, run_name)
                
                # Log Model
                mlflow.sklearn.log_model(final_model, name="model")
                
                results.append({
                    "target": target,
                    "model": model_name,
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"],
                    "r2": metrics["r2"]
                })
            
            # Update the outer progress bar after completing a full model run
            pbar.update(1)

print("\n=== All 9 LORO runs complete! Models trained on 100% data and logged to MLflow. ===")

### Model Comparison

In [ ]:
results_df = pd.DataFrame(results)

# Pivot for a clean comparison view
for metric in ["rmse", "mae", "r2"]:
    print(f"\n{'='*50}")
    print(f"  {metric.upper()} by Target × Model")
    print(f"{'='*50}")
    pivot = results_df.pivot(index="target", columns="model", values=metric)
    pivot = pivot[["Ridge", "RandomForest", "XGBoost"]]  # fix column order
    
    # Highlight min for rmse/mae, max for r2
    if metric in ["rmse", "mae"]:
        display(pivot.style.highlight_min(axis=1, color="green").format("{:.4f}"))
    else:
        display(pivot.style.highlight_max(axis=1, color="green").format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ["rmse", "mae", "r2"]):
    pivot = results_df.pivot(index="target", columns="model", values=metric)
    pivot = pivot[["Ridge", "RandomForest", "XGBoost"]]
    pivot.plot(kind="bar", ax=ax, rot=45, width=0.8)
    ax.set_title(f"{metric.upper()} Comparison", fontsize=14, fontweight='bold')
    ax.set_ylabel(metric.upper())
    ax.set_xlabel("")
    ax.legend(title="Model", loc='best')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Saved to ../reports/model_comparison.png")